# 건강 임베딩 다시 학습 (RunPod, RTX PRO 4000)

`scripts/train_embedding_gpu.py`를 차례로 실행합니다. 학습 코드는 그 스크립트에 있고, 이 노트북은 순서와 결과 확인만 합니다.

- 복사본 상담 53개의 합성 질의를 빼고, 시드 3개 × 3 epoch로 학습합니다. epoch마다 체크포인트를 남깁니다.
- epoch는 dev 상담 300개로 고릅니다. 손으로 쓴 220문항 평가는 로컬에서 합니다.
- 기록: DagsHub MLflow `health-embedding`. 체크포인트: 비공개 모델 저장소 `<사용자>/ko-sroberta-dograg-b2`.
- 학습 데이터는 AI Hub 원문이라 공유 금지입니다. 끝나면 마지막 셀로 지우고, pod 볼륨도 지웁니다.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 저장소 받기와 패키지 설치
%cd /workspace
!rm -rf dograg && git clone -q --depth 1 --branch feat/embedding-retrain-gpu https://github.com/rndigkwk/dograg.git
%cd /workspace/dograg
# RunPod 이미지의 blinker는 debian이 깔아 pip이 지우지 못한다 (mlflow가 새 버전을 요구)
!pip install -q --ignore-installed blinker
!pip install -q "sentence-transformers[train]>=5.1" mlflow python-dotenv huggingface_hub
!git log --oneline -1

In [ ]:
# 토큰: 입력 칸에 붙여 넣습니다. 화면과 노트북 파일에 남지 않습니다.
import os
from getpass import getpass
os.environ["HF_TOKEN"] = getpass("Hugging Face 토큰 (비공개 데이터 읽기 + 모델 저장소 쓰기): ").strip()
os.environ["MLFLOW_TRACKING_PASSWORD"] = getpass("DagsHub 토큰 (MLflow 기록): ").strip()
os.environ["TRAIN_WORK_DIR"] = "/workspace/embedding-retrain"
print("토큰 입력됨")

## 1. 확인 실행

작은 부분으로 몇 단계만 돌립니다(1분 안쪽). 업로드하지 않고, MLflow도 pod 안에만 남깁니다.

In [ ]:
!python scripts/train_embedding_gpu.py --smoke --no-upload --local-mlflow --seeds 13 --epochs 2 2>&1 | grep -v "it/s]"
!rm -rf /workspace/embedding-retrain

## 2. 본 학습

시드 3개 × 3 epoch. 마지막에 체크포인트 9개를 모델 저장소에 올리고 DagsHub에 기록합니다. 출력은 `/workspace/retrain.log`에도 남습니다.

In [ ]:
!python scripts/train_embedding_gpu.py 2>&1 | grep --line-buffered -v "it/s]" | tee /workspace/retrain.log

## 3. 결과

In [ ]:
import json
import pandas as pd

report = json.load(open("/workspace/embedding-retrain/upload/results.json"))
print("기준 모델 dev:", report["base"])
table = pd.DataFrame([{"checkpoint": r["name"], "seed": r["seed"], "epoch": r["epoch"], **r["dev"]} for r in report["runs"]])
display(table)
display(table.groupby("epoch")[["accuracy@1", "accuracy@10", "mrr@10"]].agg(["mean", "std"]).round(4))
print("dev로 고른 체크포인트:", report["best_by_dev"])
print(json.dumps({k: v for k, v in report.items() if k != "settings"}))  # 이 줄을 복사해 보내 주세요

## 4. 정리

업로드가 끝난 것을 확인한 뒤 실행합니다. 학습 데이터와 체크포인트를 지웁니다. 그다음 RunPod에서 pod을 멈추고 볼륨을 지웁니다.

In [ ]:
!rm -rf /workspace/embedding-retrain /workspace/dograg/data/df.csv /workspace/dograg/data/finetune /workspace/retrain.log
!rm -rf ~/.cache/huggingface
for key in ("HF_TOKEN", "MLFLOW_TRACKING_PASSWORD"):
    os.environ.pop(key, None)
print("지웠습니다")